<a href="https://colab.research.google.com/github/itsnothuy/AI-PACMAN/blob/master/Chapter_3%2C_Problem_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook shows how to solve Problem #11 in Chapter 3.

We will need both NumPy for numerical calculations and SymPy for symbolic calculations.

In [ ]:
import numpy as np
import sympy as sp

Start by defining the matrix.

In [ ]:
p12 = (1/2)*sp.exp(-3)
p11 = 1-p12
p13 = 0
p14 = 0
p21 = 1/2
p22 = 0
p23 = 1/2
p24 = 0
p31 = 0
p32 = (1/2)*sp.exp(-1)
p34 = 1/2
p33 = 1-(p32+p34)
p41 = 0
p42 = 0
p43 = (1/2)*sp.exp(-1)
p44 = 1 - p43

Initially, define the matrix $P$ as a SymPy matrix.  Ultimately, we will try to find the null space of $P-I$, but singular matrices are like isolated points (actually, sets of measure 0) scattered through the continuum of matrix space, so if there's any rounding, we will (almost certainly) no longer have a singular matrix.  If you try to do the following computations in NumPy, you will get a trivial null space.   

In [ ]:
P = sp.Matrix([[p11, p12, p13, p14], [p21, p22, p23, p24], [p31, p32, p33, p34], [p41, p42, p43, p44]])
P

Matrix([
[1 - 0.5*exp(-3), 0.5*exp(-3),                 0,               0],
[            0.5,           0,               0.5,               0],
[              0, 0.5*exp(-1), 0.5 - 0.5*exp(-1),             0.5],
[              0,           0,       0.5*exp(-1), 1 - 0.5*exp(-1)]])

Subtract the identity matrix.

In [ ]:
Q = P-sp.eye(4)
Q

Matrix([
[-0.5*exp(-3), 0.5*exp(-3),                  0,            0],
[         0.5,          -1,                0.5,            0],
[           0, 0.5*exp(-1), -0.5 - 0.5*exp(-1),          0.5],
[           0,           0,        0.5*exp(-1), -0.5*exp(-1)]])

SymPy calculates null spaces for multiplying with the matrix on the left, so we'll take the transpose of the whole situation, do the computations, and the transpose the result to return to the convention in our book.

In [ ]:
Q.T

Matrix([
[-0.5*exp(-3), 0.5,                  0,            0],
[ 0.5*exp(-3),  -1,        0.5*exp(-1),            0],
[           0, 0.5, -0.5 - 0.5*exp(-1),  0.5*exp(-1)],
[           0,   0,                0.5, -0.5*exp(-1)]])

Here's the null space.

In [ ]:
NS = Q.T.nullspace()
NS

[Matrix([
 [      1.0*E],
 [1.0*exp(-2)],
 [1.0*exp(-1)],
 [          1]])]

Actually, we want the transpose.

In [ ]:
NS[0].T

Matrix([[1.0*E, 1.0*exp(-2), 1.0*exp(-1), 1]])

We need to rescale so that this is a probability vector.  

In [ ]:
S = NS[0][0]+NS[0][1]+NS[0][2]+NS[0][3]
NS = NS[0].T/S
NS

Matrix([[1.0*E/(1.0*exp(-2) + 1.0*exp(-1) + 1 + 1.0*E), 1.0*exp(-2)/(1.0*exp(-2) + 1.0*exp(-1) + 1 + 1.0*E), 1.0*exp(-1)/(1.0*exp(-2) + 1.0*exp(-1) + 1 + 1.0*E), 1/(1.0*exp(-2) + 1.0*exp(-1) + 1 + 1.0*E)]])

Notice that these are exactly the values given at the end of the problem.  These values are the "energized" versions of the outputs of $f$.  In other words, $f$ is an energy function, and energizing it turns it into an unnormalized distribution.  The matrix $P$ is, evidently, a Markov process designed so that the energized version of $f$ is its invariant distribution (up to rescaling).  You can actually reconstruct the transition probabilities from the proposal process $g$ by looking at the rows of the matrix $P$.

Here;s a floating point version as a NumPy matrix.

In [ ]:
nv = np.array(NS).astype(np.float64)
nv

array([[0.64391426, 0.0320586 , 0.08714432, 0.23688282]])

Now let's confirm by repeated multiplication (exponentiation).  We have to be careful to get floating point numbers when converting P (a SymPy matrix) to NP (a NumPy matrix).  That's what ```.astype(np.float64)``` does.



In [ ]:
NP = np.matrix(P).astype(np.float64)
NP

matrix([[0.97510647, 0.02489353, 0.        , 0.        ],
        [0.5       , 0.        , 0.5       , 0.        ],
        [0.        , 0.18393972, 0.31606028, 0.5       ],
        [0.        , 0.        , 0.18393972, 0.81606028]])

In [ ]:
v = np.matrix([0, 1, 0, 0])
v

matrix([[0, 1, 0, 0]])

In [ ]:
for i in range(100):
    print(v*NP**i)


[[0. 1. 0. 0.]]
[[0.5 0.  0.5 0. ]]
[[0.48755323 0.10441663 0.15803014 0.25      ]]
[[0.52762462 0.04120494 0.14814029 0.28303014]]
[[0.53509265 0.04038333 0.11948422 0.3050398 ]]
[[0.54196397 0.03529824 0.11406481 0.30867298]]
[[0.54612169 0.03447245 0.1104777  0.30892816]]
[[0.54976302 0.03391614 0.108978   0.30734285]]
[[0.55303554 0.03373093 0.10793424 0.30529929]]
[[0.55613399 0.0336204  0.10713586 0.30310975]]
[[0.55910006 0.03355068 0.10642551 0.30092375]]
[[0.56195742 0.03349386 0.10576405 0.29878468]]
[[0.56471524 0.03344332 0.10513311 0.29670833]]
[[0.56737914 0.03339591 0.10452651 0.29469844]]
[[0.56995302 0.03335065 0.10394138 0.29275494]]
[[0.5724402  0.03330709 0.10337633 0.29087637]]
[[0.57484369 0.03326507 0.10283042 0.28906082]]
[[0.57716634 0.03322449 0.10230291 0.28730626]]
[[0.57941087 0.03318528 0.10179317 0.28561068]]
[[0.58157993 0.03314739 0.10130057 0.28397212]]
[[0.58367604 0.03311078 0.10082453 0.28238865]]
[[0.58570167 0.0330754  0.10036451 0.28085843]]
[[0.

In [ ]:
v*NP**1000

matrix([[0.64391426, 0.0320586 , 0.08714432, 0.23688282]])